In [ ]:
# Core libraries
import glob
import re
import datetime as dt
import warnings
from itertools import combinations
from tqdm import tqdm  # Import tqdm for progress bars

# Numerical and data manipulation libraries
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Financial data analysis
import yfinance as yf

# Statistical analysis
import statsmodels.api as sm
from scipy.stats import pearsonr
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.regression.linear_model import OLS

# Time series forecasting
import pmdarima as pm

# Machine Learning imports
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, roc_auc_score,
    mean_absolute_error, mean_squared_error, mean_absolute_percentage_error,
    ConfusionMatrixDisplay, roc_curve, auc
)

# XGBoost
import xgboost as xgb
from xgboost import XGBRFClassifier

# Deep Learning
import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM, Dense

# Suppress warnings
warnings.filterwarnings("ignore")

def hac_variance(daily_returns, lag_truncation=None):
    """
    Calculate the HAC variance estimator.
    
    Parameters:
        daily_returns (array-like): Daily returns series.
        lag_truncation (int): Lag truncation parameter (K). Defaults to floor(n^0.25).
    
    Returns:
        float: HAC variance estimate.
    """
    n = len(daily_returns)
    mean_return = np.mean(daily_returns)
    
    # Default lag truncation K = floor(n^(1/4))
    if lag_truncation is None:
        lag_truncation = int(np.floor(0.75*(n**(1/3))))
    
    # Compute gamma_0 (variance)
    gamma_0 = np.mean((daily_returns - mean_return)**2)
    
    # Compute gamma_k (covariances) and weighted sum
    gamma_k_sum = 0
    for k in range(1, lag_truncation):
        # Compute sample covariance for lag k
        cov_k = np.mean(
            (daily_returns[k:] - mean_return) * (daily_returns[:-k] - mean_return)
        )
        # Weight by (1 - k/K)
        weight = 1 - k / lag_truncation
        gamma_k_sum += weight * cov_k
    
    # HAC variance
    hac_variance = gamma_0 + 2 * gamma_k_sum
    return hac_variance

def sharpe_ratio_hac(daily_returns, annualization_factor=252):
    """
    Calculate the Sharpe ratio using the HAC variance estimator.
    
    Parameters:
        daily_returns (array-like): Daily returns series.
        annualization_factor (int): Factor for annualizing the Sharpe ratio (e.g., 252 for daily data).
    
    Returns:
        float: Sharpe ratio using HAC variance estimator.
    """
    mean_return = np.mean(daily_returns)
    hac_var = hac_variance(daily_returns)
    hac_std = np.sqrt(hac_var)
    
    sharpe_ratio_hac = mean_return / hac_std * np.sqrt(annualization_factor)
    return sharpe_ratio_hac

# Utility functions
def clean_df(df):
    """Cleans and preprocesses the input DataFrame."""
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df.dropna(subset=['date'], inplace=True)
    df['date'] = df['date'].dt.date
    df.drop_duplicates(inplace=True)
    #df = filter_single_crypto_comments(df)
    df.sort_values(by='date', inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

def daily_sentiment(df, index='simple',correction='none', model='gpt',window=5, alpha=0.1, volume_adjustment=False, sd=False):
    """Calculates daily sentiment with various correction options."""
    # Create a new dataframe to store daily sentiment index
    daily_stats = pd.DataFrame()
    
    if index in ['simple', 'no_neutral','ln','ratio','disagreement']:
        if index == 'simple':
            # Calculate daily sentiment based on bullish, bearish, and neutral
            daily_counts = df.groupby('date').apply(
                lambda group: pd.Series({
                    'bullish_n': (group[f'prediction_{model}'] == 'bullish').sum(),
                    'bearish_n': (group[f'prediction_{model}'] == 'bearish').sum(),
                    'neutral_n': (group[f'prediction_{model}'] == 'neutral').sum()
                })
            ).reset_index()
            
            daily_counts['daily_sentiment_index'] = (
                daily_counts['bullish_n'] - daily_counts['bearish_n']
            ) / (
                daily_counts['bullish_n'] + daily_counts['bearish_n'] + daily_counts['neutral_n']
            )
        
        elif index == 'no_neutral':
            # Calculate daily sentiment based on bullish and bearish only
            daily_counts = df.groupby('date').apply(
                lambda group: pd.Series({
                    'bullish_n': (group[f'prediction_{model}'] == 'bullish').sum(),
                    'bearish_n': (group[f'prediction_{model}'] == 'bearish').sum()
                })
            ).reset_index()
            
            daily_counts['daily_sentiment_index'] = (
                daily_counts['bullish_n'] - daily_counts['bearish_n']
            ) / (
                daily_counts['bullish_n'] + daily_counts['bearish_n']
            )
        
        elif index == 'ln':
            # Calculate daily sentiment based on bullish, bearish, and neutral
            daily_counts = df.groupby('date').apply(
                lambda group: pd.Series({
                    'bullish_n': (group[f'prediction_{model}'] == 'bullish').sum(),
                    'bearish_n': (group[f'prediction_{model}'] == 'bearish').sum(),
                    'neutral_n': (group[f'prediction_{model}'] == 'neutral').sum()
                })
            ).reset_index()
            
            daily_counts['daily_sentiment_index'] = np.log(
                (1+daily_counts['bullish_n'])/(1+daily_counts['bearish_n']))
    
        elif index == 'ratio':
            # Calculate daily sentiment based on bullish, bearish, and neutral
            daily_counts = df.groupby('date').apply(
                lambda group: pd.Series({
                    'bullish_n': (group[f'prediction_{model}'] == 'bullish').sum(),
                    'bearish_n': (group[f'prediction_{model}'] == 'bearish').sum(),
                    'neutral_n': (group[f'prediction_{model}'] == 'neutral').sum()
                })
            ).reset_index()
            
            daily_counts['daily_sentiment_index'] = daily_counts['bullish_n']/daily_counts['bearish_n']
        
        elif index == 'disagreement':
            # Calculate daily sentiment based on bullish, bearish, and neutral
            daily_counts = df.groupby('date').apply(
                lambda group: pd.Series({
                    'bullish_n': (group[f'prediction_{model}'] == 'bullish').sum(),
                    'bearish_n': (group[f'prediction_{model}'] == 'bearish').sum(),
                    'neutral_n': (group[f'prediction_{model}'] == 'neutral').sum()
                })
            ).reset_index()
            
            IS = (
                daily_counts['bullish_n'] - daily_counts['bearish_n']
            ) / (
                daily_counts['bullish_n'] + daily_counts['bearish_n']
            )            
            
            DOS = np.sqrt(1-IS**2)
            
            daily_counts['daily_sentiment_index'] = 1-DOS
        
        daily_stats = daily_counts[['date', 'daily_sentiment_index']]
        daily_stats['count'] = df.groupby('date').size().values        
        
        
    
    # Apply corrections based on the base sentiment index
    if sd:
        sentiment_index = daily_stats['daily_sentiment_index'].std()
    else:
        sentiment_index = daily_stats['daily_sentiment_index']
    
    if volume_adjustment:
        daily_stats['sentiment_index'] = sentiment_index * daily_stats['count'] / daily_stats['count'].sum()
    else:
        daily_stats['sentiment_index'] = sentiment_index
    
    if correction == 'z_score':
        daily_stats['sentiment_index'] = (daily_stats['sentiment_index'] - 
                                          daily_stats['sentiment_index'].mean()) / daily_stats['sentiment_index'].std()
    elif correction == 'moving_average':
        daily_stats['sentiment_index'] -= daily_stats['sentiment_index'].rolling(window=window, min_periods=1).mean()
    elif correction == 'ewma':
        daily_stats['sentiment_index'] = daily_stats['sentiment_index'].ewm(alpha=alpha).mean()
    elif correction == 'dewma':
        ema_baseline = daily_stats['sentiment_index'].ewm(alpha=alpha).mean()
        daily_stats['sentiment_index'] -= ema_baseline
    elif correction == 'dema':
        ema1 = daily_stats['sentiment_index'].ewm(alpha=alpha).mean()
        ema2 = ema1.ewm(alpha=alpha).mean()
        daily_stats['sentiment_index'] = 2 * ema1 - ema2
    elif correction == 'roc_ewma':
        daily_stats['sentiment_index'] = daily_stats['sentiment_index'].diff().ewm(alpha=alpha).mean()
    
    daily_stats['cumulative_sentiment_index'] = daily_stats['sentiment_index'].cumsum()
    return daily_stats[['date', 'sentiment_index', 'cumulative_sentiment_index','count']]

def download_and_prepare_data(ticker, start_date, end_date):
    """Downloads and prepares financial data for analysis."""
    data = yf.download(ticker, start=start_date, end=end_date, progress=False)
    data['logreturn'] = np.log(data['Adj Close']) - np.log(data['Adj Close']).shift(1)
    data['logprice'] = np.log(data['Adj Close'])
    data['total_asset_volume'] = data['Volume'] * data['Adj Close']
    data.reset_index(inplace=True)
    return data

def calculate_sentiment_metrics(df, sentiment_df):
    """Calculates sentiment-based metrics and merges with financial data."""
    sentiment_df['Date'] = pd.to_datetime(sentiment_df['date'])
    merged_df = pd.merge(df, sentiment_df, on='Date', how='inner')
    
    #Positive logreturn
    merged_df['positive_logreturn'] = merged_df['logreturn'].apply(lambda x: 1 if x > 0 else 0)
    merged_df['positive_sentiment'] = merged_df['sentiment_index'].apply(lambda x: 1 if x > 0 else 0)


    # Add sentiment streaks
    merged_df['positive_sentiment_days'] = (merged_df['sentiment_index'] > 0).astype(int).groupby((merged_df['sentiment_index'] <= 0).cumsum()).cumsum()
    merged_df['negative_sentiment_days'] = (merged_df['sentiment_index'] < 0).astype(int).groupby((merged_df['sentiment_index'] >= 0).cumsum()).cumsum()

    # Add log return streaks
    merged_df['positive_logreturn_days'] = (merged_df['logreturn'] > 0).astype(int).groupby((merged_df['logreturn'] <= 0).cumsum()).cumsum()
    merged_df['negative_logreturn_days'] = (merged_df['logreturn'] < 0).astype(int).groupby((merged_df['logreturn'] >= 0).cumsum()).cumsum()
    
    # Add sentiment momentum
    for days in [2, 3, 5, 7]:
        merged_df[f'sentiment_momentum_{days}d'] = merged_df['sentiment_index'] - merged_df['sentiment_index'].shift(days)

#     merged_df = create_ma(merged_df, ['sentiment_index'], 25, 50)
    
    return merged_df

def visualize_correlation(asset, ticker, merged_df,sentiment_column='cumulative_sentiment_index'):
    """Visualizes the correlation between price and sentiment."""
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.set_xlabel('Date')
    ax1.set_ylabel(ticker + ' Adjusted Close Price', color='blue')
    line1, = ax1.plot(merged_df['Date'], merged_df['Adj Close'], color='blue', label='Adj Close')
    ax1.tick_params(axis='y', labelcolor='blue')
    ax2 = ax1.twinx()
    ax2.set_ylabel('Sentiment Index', color='red')
    line2, = ax2.plot(merged_df['Date'], merged_df[sentiment_column], color='red', label='Sentiment Index')
    #line3, = ax2.plot(merged_df['Date'], merged_df['cumulative_sentiment_index'], color='orange', label='Cumulative Sentiment Index')
    ax2.tick_params(axis='y', labelcolor='red')
    lines = [line1, line2]
    labels = [line.get_label() for line in lines]
    fig.legend(lines, labels, loc="upper left", bbox_to_anchor=(0.1, 0.85))
    fig.suptitle(asset)
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

def create_ma(df, columns, short_window, long_window):
    """
    Create moving averages for specified columns in the DataFrame.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        columns (list): A list of column names to calculate moving averages for.
        short_window (int): The window size for the short-term moving average.
        long_window (int): The window size for the long-term moving average.
    
    Returns:
        pd.DataFrame: DataFrame with added moving average columns.
    """
    for col in columns:
        df[f'short_ma_{col}'] = df[col].rolling(window=short_window).mean()
        df[f'long_ma_{col}'] = df[col].rolling(window=long_window).mean()
    return df

def plot_trading_strategy(merged_df, underlying_column, short_window, long_window, buy_signals, sell_signals, buy_prices, sell_prices):
    """
    Plot trading signals and sentiment-driven moving averages.
    """
    # Create figure and primary y-axis for adjusted close price
    fig, ax1 = plt.subplots(figsize=(12, 8))
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Adjusted Close Price', color='blue')
    ax1.plot(merged_df['Date'], merged_df['Adj Close'], color='blue', label='Adj Close', zorder=1)
    ax1.tick_params(axis='y', labelcolor='blue')
    
    if underlying_column == 'Adj Close':
        ax1.plot(merged_df['Date'], merged_df[f'short_ma_{underlying_column}'], color='green', label=f'{short_window}-day MA', zorder=1)
        ax1.plot(merged_df['Date'], merged_df[f'long_ma_{underlying_column}'], color='red', label=f'{long_window}-day MA', zorder=1)
        
    else:
        ax2 = ax1.twinx()
        ax2.set_ylabel('Sentiment Index', color='darkorange')
        ax2.plot(merged_df['Date'], merged_df[underlying_column], color='darkorange', label='Sentiment Index', zorder=1)
        ax2.plot(merged_df['Date'], merged_df[f'short_ma_{underlying_column}'], color='green', label=f'{short_window}-day MA', zorder=1)
        ax2.plot(merged_df['Date'], merged_df[f'long_ma_{underlying_column}'], color='red', label=f'{long_window}-day MA', zorder=1)
        ax2.tick_params(axis='y', labelcolor='darkorange')

    # Plot buy and sell signals
    ax1.scatter(buy_signals, buy_prices, color='green', label='Buy Signal', s=50, zorder=5)
    ax1.scatter(sell_signals, sell_prices, color='red', label='Sell Signal', s=50, zorder=5)

    # Add vertical lines for buy and sell signals
    for date in buy_signals:
        ax1.axvline(date, color='green', linestyle='--', alpha=0.5, label='Buy Signal', zorder=2)
    for date in sell_signals:
        ax1.axvline(date, color='red', linestyle='--', alpha=0.5, label='Sell Signal', zorder=2)

    # Combine legend handles from both axes
    handles1, labels1 = ax1.get_legend_handles_labels()
    if underlying_column != 'Adj Close':
        handles2, labels2 = ax2.get_legend_handles_labels()
        handles = handles1 + handles2
        labels = labels1 + labels2
        unique_handles_labels = dict(zip(labels, handles))
        fig.legend(unique_handles_labels.values(), unique_handles_labels.keys(), loc="upper left", bbox_to_anchor=(0.1, 0.85))
    else:
        unique_handles_labels = dict(zip(labels1, handles1))
        fig.legend(unique_handles_labels.values(), unique_handles_labels.keys(), loc="upper left", bbox_to_anchor=(0.1, 0.85))
        
    # Set title and adjust layout
    fig.suptitle(f"Trading Strategy with Signals")
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to make space for legend

    plt.show()
    
def moving_average_strategy(merged_df, underlying_column, short_window, long_window, initial_cash=1000, plot=False):
    short_ma_col = f'short_ma_{underlying_column}'
    long_ma_col = f'long_ma_{underlying_column}'

    if short_ma_col not in merged_df.columns or long_ma_col not in merged_df.columns:
        merged_df = create_ma(merged_df, [underlying_column], short_window, long_window)

    # Initialize variables for tracking trades
    last_signal = 'sell'
    buy_signals, sell_signals = [], []
    buy_prices, sell_prices = [], []
    cash = initial_cash
    position = 0
    transaction_log = []

    # Simulate trades
    for idx, row in merged_df.iterrows():
        date = row['Date']
        adj_close_price = row['Adj Close']
        short_ma = row[short_ma_col]
        long_ma = row[long_ma_col]

        if not pd.isna(short_ma) and not pd.isna(long_ma):
            # Buy signal: short MA crosses above long MA
            if short_ma > long_ma and last_signal == 'sell':
                buy_signals.append(date)
                buy_prices.append(adj_close_price)
                position = cash / adj_close_price  # Buy as many shares as cash allows
                cash = 0  # All cash converted to position
                transaction_log.append({'date': date, 'type': 'buy', 'price': adj_close_price, 'position': position})
                last_signal = 'buy'

            # Sell signal: short MA crosses below long MA
            elif short_ma < long_ma and last_signal == 'buy':
                sell_signals.append(date)
                sell_prices.append(adj_close_price)
                cash = position * adj_close_price  # Sell all shares
                transaction_log.append({'date': date, 'type': 'sell', 'price': adj_close_price, 'position': position})
                position = 0  # Clear position
                last_signal = 'sell'

        # Update portfolio value for every day
        portfolio_value = cash + (position * adj_close_price if position > 0 else 0)
        merged_df.loc[idx, f'portfolio_value_{underlying_column}'] = portfolio_value

    # Final portfolio value
    final_price = merged_df.iloc[-1]['Adj Close']
    final_portfolio_value = cash + (position * final_price if position > 0 else 0)

    # Calculate daily returns based on portfolio value
    merged_df[f'daily_returns_{underlying_column}'] = merged_df[f'portfolio_value_{underlying_column}'].pct_change().fillna(0)

    # Calculate normal Sharpe ratio
    daily_returns = merged_df[f'daily_returns_{underlying_column}']
    normal_sharpe_ratio = daily_returns.mean() / daily_returns.std() * np.sqrt(252)  # Annualized Sharpe ratio

    
    #sharpe_ratio_hac = sharpe_ratio_hac(daily_returns)
    std_error_hac = np.sqrt(hac_variance(daily_returns))
    sharpe_ratio_hac = daily_returns.mean() / (std_error_hac) * np.sqrt(252)
    
    start_date = merged_df['Date'].iloc[0]
    end_date = merged_df['Date'].iloc[-1]
    time_horizon_years = (end_date - start_date).days / 365.25

    # Calculate annualized return
    total_return = (final_portfolio_value - initial_cash) / initial_cash
    annualized_return = (1 + total_return) ** (1 / time_horizon_years) - 1
    
    
    # Extract the HAC standard error of the mean

    # Consolidating metrics for comparison
    strategies_metrics = {
        'net_profit': final_portfolio_value - initial_cash,
        'return_%': total_return * 100,
        'annualized_return_%': annualized_return * 100,
        'sharpe_ratio': normal_sharpe_ratio,
        'sharpe_ratio_hac': sharpe_ratio_hac,
        'total_trades': len(transaction_log),
    }

    # Plotting if requested
    if plot:
        plot_trading_strategy(merged_df, underlying_column, short_window, long_window, buy_signals, sell_signals, buy_prices, sell_prices)

    # Return results
    return merged_df, strategies_metrics

def buy_and_hold_strategy(merged_df, underlying_column, initial_cash=1000):
    # Buy at the first available price and hold
    first_price = merged_df.iloc[0][underlying_column]
    position = initial_cash / first_price
    cash = 0
    transaction_log = [{'date': merged_df.iloc[0]['Date'], 'type': 'buy', 'price': first_price, 'position': position}]

    # Portfolio value will be position * current price for every day
    portfolio_values = []
    for idx, row in merged_df.iterrows():
        adj_close_price = row['Adj Close']
        portfolio_value = position * adj_close_price  # Always hold the same amount of asset
        portfolio_values.append(portfolio_value)

    merged_df[f'portfolio_value_buy_hold'] = portfolio_values

    # Final portfolio value
    final_price = merged_df.iloc[-1]['Adj Close']
    final_portfolio_value = position * final_price

    # Calculate daily returns
    merged_df[f'daily_returns_buy_hold'] = merged_df[f'portfolio_value_buy_hold'].pct_change().fillna(0)

    # Calculate Sharpe ratio
    daily_returns = merged_df[f'daily_returns_buy_hold']
    sharpe_ratio = daily_returns.mean() / daily_returns.std() * np.sqrt(252)

    
    #sharpe_ratio_hac = sharpe_ratio_hac(daily_returns)
    std_error_hac = np.sqrt(hac_variance(daily_returns))
    sharpe_ratio_hac = daily_returns.mean() / (std_error_hac) * np.sqrt(252)
    
    start_date = merged_df['Date'].iloc[0]
    end_date = merged_df['Date'].iloc[-1]
    time_horizon_years = (end_date - start_date).days / 365.25

    # Calculate annualized return
    total_return = (final_portfolio_value - initial_cash) / initial_cash
    annualized_return = (1 + total_return) ** (1 / time_horizon_years) - 1

    strategies_metrics = {
        'net_profit': final_portfolio_value - initial_cash,
        'return_%': total_return * 100,
        'annualized_return_%': annualized_return * 100,
        'sharpe_ratio': sharpe_ratio,
        'sharpe_ratio_hac': sharpe_ratio_hac,
        'total_trades': len(transaction_log),
    }
    


    return merged_df, strategies_metrics



def run_and_store_metrics(strategy_name, merged_df, underlying_column, short_window, long_window, plot=False):
    if strategy_name == "Buy_and_Hold":
        # Run Buy and Hold strategy
        merged_df, strategies_metrics = buy_and_hold_strategy(merged_df, underlying_column, initial_cash=1000)
    elif strategy_name == 'Oscillator':
        # Run Buy and Hold strategy
        merged_df, strategies_metrics = sentiment_oscillator_strategy(merged_df, sentiment_column, underlying_column, overbought_threshold=0.8, oversold_threshold=0.2, initial_cash=1000, plot=False)
    else:
        # Run Moving Average strategy
        merged_df, strategies_metrics = moving_average_strategy(merged_df, underlying_column, short_window, long_window, initial_cash=1000, plot=plot)
    
    # Store the metrics in a global dictionary
    if strategy_name not in metrics_dict:
        metrics_dict[strategy_name] = strategies_metrics

    #return merged_df, metrics_dict
    
def cointegration_testing(df, col_x, col_y):
    """
    Perform comprehensive cointegration and causality analysis between two variables.

    Parameters:
    df (pd.DataFrame): DataFrame containing the data.
    col_x (str): Name of the first variable column (e.g., logprice).
    col_y (str): Name of the second variable column (e.g., market sentiment).

    Returns:
    None: Prints detailed analysis results including cointegration, stationarity, VECM summary, Granger causality, and Pearson correlation.
    """
    
    # Create a DataFrame with the input variables
    coint_df = df[[col_x, col_y]].copy()
    coint_df.dropna(inplace=True)

    # Perform Pearson correlation test
    corr, p_value = pearsonr(coint_df[col_x], coint_df[col_y])
    print(f"\nPearson Correlation between {col_x} and {col_y}: {corr:.4f}")
    print(f"p-value: {p_value:.4f}")

    # Perform Johansen cointegration test
    res_coint = coint_johansen(coint_df, det_order=0, k_ar_diff=3)  # k_ar_diff=p-1, the number of lags in first-differenced model
    print("\nJohansen Cointegration Test Results:")
    print("Trace statistic:", res_coint.lr1)
    print("Critical values for trace statistic:", res_coint.cvt)
    print("Maximum eigenvalue statistic:", res_coint.lr2)
    print("Critical values for maximum eigenvalue statistic:", res_coint.cvt)

    # Determine the number of cointegrated relations
    trace_crit = res_coint.cvt[:, 1]  # 5% critical values for trace
    max_eig_crit = res_coint.cvt[:, 1]  # 5% critical values for max eigenvalue
    n_cointegrated = sum(res_coint.lr1 > trace_crit)
    print(f"Number of cointegrated relations (trace test): {n_cointegrated}")

    # Fit a Vector Error Correction Model (VECM)
    mod = VECM(coint_df)
    res = mod.fit()
    print("\nVECM Summary:")
    print(res.summary())

    # Perform ADF test for stationarity on both series with 5 lags
    print("\nADF Test Results:")
    for col in [col_x, col_y]:
        adf_result = adfuller(coint_df[col], maxlag=5)
        print(f"{col}: p-value = {adf_result[1]} {'(Stationary)' if adf_result[1] < 0.05 else '(Non-stationary)'}")

    # Create first differences
    coint_df[f'{col_x}_diff'] = coint_df[col_x].diff()
    coint_df[f'{col_y}_diff'] = coint_df[col_y].diff()
    coint_df.dropna(inplace=True)

    # Perform Granger causality tests
    print("\nGranger Causality Test Results:")
    print(f"Testing causality: {col_y} -> {col_x}")
    gc_res1 = grangercausalitytests(coint_df[[f'{col_x}', f'{col_y}']], maxlag=10, verbose=True)
    print(f"\nTesting causality: {col_x} -> {col_y}")
    gc_res2 = grangercausalitytests(coint_df[[f'{col_y}', f'{col_x}']], maxlag=10, verbose=True)
                                    
def create_diff_lags(df,diff=True,length=15):
    
    if diff:
        for column in df.columns:
            if column not in ['date','Date','FG_class','positive_sentiment_days','negative_sentiment_days','positive_logreturn']:
                df[f'{column}_diff'] = df[column].diff()  # Create lag of 1
        
    for column in df.columns:
        if column not in ['date','Date','FG_class','positive_logreturn','positive_sentiment']:
            for i in range(1,length+1):
                df[f'{column}_lag{i}'] = df[column].shift(i)  # Create lag of 1
    df.dropna(inplace=True)
            
    return df

def evaluate_models_lags():
    # Initialize dictionaries to store results
    results = {
        'Logistic Regression': {
            'best_combination': None,
            'best_accuracy': 0
        },
        'SVM': {
            'best_combination': None,
            'best_accuracy': 0
        },
        'XGBoost-RF': {
            'best_combination': None,
            'best_accuracy': 0
        }
    }

    # Calculate the split indices for a 60-20-20 split
    train_size = int(0.6 * len(filtered_df))
    val_size = int(0.2 * len(filtered_df))  # 20% for validation
    test_size = len(filtered_df) - train_size - val_size  # Remaining 20% for testing

    # Track how many combinations are checked
    total_combinations = 0

    # Calculate the total number of combinations
    for r in range(1, 6):  # Loop through different feature counts (1 to 5)
        if r == 1:
            # If only one feature is used, it must be a sentiment feature
            total_combinations += len(sentiment_features)
        else:
            # Determine how many sentiment lags are needed
            sentiment_count = max(1, r // 2)  # At least one sentiment feature
            total_combinations += len(list(combinations(sentiment_features, sentiment_count))) * \
                                   len(list(combinations(other_features, r - sentiment_count)))

    # Loop through combinations of sentiment and other features using tqdm for progress
    with tqdm(total=total_combinations) as pbar:  # Initialize tqdm progress bar
        for r in range(1, 6):  # Change the range depending on the max number of features
            if r == 1:
                # Special case: only one feature, must be a sentiment feature
                for sentiment_feature in sentiment_features:
                    combo = (sentiment_feature,)
                    X = filtered_df[list(combo)]

                    # Sequential Train and Validation split
                    X_train = X[:train_size]
                    y_train = y[:train_size]

                    X_val = X[train_size:train_size + val_size]
                    y_val = y[train_size:train_size + val_size]

                    # Evaluate models with the single sentiment feature
                    results = evaluate_single_combination(
                        X_train, y_train, X_val, y_val, combo, results
                    )
                    pbar.update(1)  # Update progress bar by 1 step
            else:
                # Determine how many sentiment lags are needed
                sentiment_count = max(1, r // 2)  # At least one sentiment feature
                for sentiment_combo in combinations(sentiment_features, sentiment_count):
                    for other_combo in combinations(other_features, r - sentiment_count):
                        combo = sentiment_combo + other_combo
                        X = filtered_df[list(combo)]

                        # Sequential Train and Validation split
                        X_train = X[:train_size]
                        y_train = y[:train_size]

                        X_val = X[train_size:train_size + val_size]
                        y_val = y[train_size:train_size + val_size]

                        # Evaluate models with the current combination
                        results = evaluate_single_combination(
                            X_train, y_train, X_val, y_val, combo, results
                        )
                        pbar.update(1)  # Update progress bar by 1 step

    return results

def evaluate_single_combination(X_train, y_train, X_val, y_val, combo, results):
    """
    Helper function to evaluate a single combination of features for all models.
    """
    # Logistic Regression
    logreg_model = LogisticRegression(max_iter=1000, random_state=42)
    logreg_model.fit(X_train, y_train)
    y_pred_logreg = logreg_model.predict(X_val)
    accuracy_logreg = accuracy_score(y_val, y_pred_logreg)
    if accuracy_logreg > results['Logistic Regression']['best_accuracy']:
        results['Logistic Regression']['best_accuracy'] = accuracy_logreg
        results['Logistic Regression']['best_combination'] = combo

    # Support Vector Machine (SVM)
    svm_model = SVC(kernel='linear', random_state=42)
    svm_model.fit(X_train, y_train)
    y_pred_svm = svm_model.predict(X_val)
    accuracy_svm = accuracy_score(y_val, y_pred_svm)
    if accuracy_svm > results['SVM']['best_accuracy']:
        results['SVM']['best_accuracy'] = accuracy_svm
        results['SVM']['best_combination'] = combo

    # XGBoost-RF
    xgbrf_model = XGBRFClassifier(n_estimators=100, random_state=42, use_label_encoder=False)
    xgbrf_model.fit(X_train, y_train)
    y_pred_xgbrf = xgbrf_model.predict(X_val)
    accuracy_xgbrf = accuracy_score(y_val, y_pred_xgbrf)
    if accuracy_xgbrf > results['XGBoost-RF']['best_accuracy']:
        results['XGBoost-RF']['best_accuracy'] = accuracy_xgbrf
        results['XGBoost-RF']['best_combination'] = combo

    return results



def xgbrf_best(X, y):
    # Split the data for time series
    train_size = int(0.6 * len(X))  # 60% for training
    val_size = int(0.2 * len(X))    # 20% for validation
    test_size = len(X) - train_size - val_size  # Remaining 20% for testing

    # Sequential splits
    X_train_val = X[:train_size + val_size]
    y_train_val = y[:train_size + val_size]

    X_test = X[train_size + val_size:]
    y_test = y[train_size + val_size:]

    # Define the XGBoost model
    xgb_rf = xgb.XGBClassifier(
        booster='dart',  # Use 'dart' to simulate Random Forest behavior in XGBoost
        objective='binary:logistic',  # Binary classification
        eval_metric='logloss'  # Log loss as the evaluation metric
    )

    # Set up the grid of hyperparameters for tuning
    param_grid = {
        'n_estimators': [50, 100, 200],  # Number of trees
        'max_depth': [3, 5, 7],  # Depth of trees
        'learning_rate': [0.01, 0.05, 0.1],  # Learning rate
        'subsample': [0.7, 0.8, 1.0],  # Proportion of data used for each boosting round
        'colsample_bytree': [0.7, 0.8, 1.0],  # Column subsampling
        'gamma': [0, 0.1, 0.2],  # Regularization term to control overfitting
    }

    # TimeSeriesSplit for time series validation
    tscv = TimeSeriesSplit(n_splits=5)

    # Use GridSearchCV for hyperparameter tuning
    grid_search = GridSearchCV(
        estimator=xgb_rf,
        param_grid=param_grid,
        scoring='accuracy',  # Use accuracy for evaluation
        cv=tscv,  # Time-series split for validation
        verbose=1,
        n_jobs=-1  # Parallelize for faster computation
    )

    # Fit the model with training and validation data
    grid_search.fit(X_train_val, y_train_val)

    # Get the best model from grid search
    best_model = grid_search.best_estimator_

    # Predict on the test set using the fine-tuned model
    y_pred = best_model.predict(X_test)

    # Evaluate the model's accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Calculate ROC AUC score
    y_prob = best_model.predict_proba(X_test)[:, 1]  # Probability estimates for the positive class
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    return accuracy, roc_auc

def svm_best(X, y):
    # Split the data for time series
    train_size = int(0.6 * len(X))  # 60% for training
    val_size = int(0.2 * len(X))    # 20% for validation
    test_size = len(X) - train_size - val_size  # Remaining 20% for testing

    # Sequential splits
    X_train_val = X[:train_size + val_size]
    y_train_val = y[:train_size + val_size]

    X_test = X[train_size + val_size:]
    y_test = y[train_size + val_size:]

    # Define the SVM model
    svm_model = SVC(probability=True, kernel='rbf')  # rbf kernel for SVM

    # Set up the grid of hyperparameters for tuning
    param_grid = {
        'C': [0.1, 1, 10],  # Regularization parameter
        'gamma': ['scale', 'auto', 0.1, 0.01],  # Kernel coefficient
        'kernel': ['rbf', 'linear'],  # Kernel types
    }

    # TimeSeriesSplit for time series validation
    tscv = TimeSeriesSplit(n_splits=5)

    # Use GridSearchCV for hyperparameter tuning
    grid_search = GridSearchCV(
        estimator=svm_model,
        param_grid=param_grid,
        scoring='accuracy',  # Use accuracy for evaluation
        cv=tscv,  # Time-series split for validation
        verbose=1,
        n_jobs=-1  # Parallelize for faster computation
    )

    # Fit the model with training and validation data
    grid_search.fit(X_train_val, y_train_val)

    # Get the best model from grid search
    best_model = grid_search.best_estimator_

    # Predict on the test set using the fine-tuned model
    y_pred = best_model.predict(X_test)

    # Evaluate the model's accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Calculate ROC AUC score
    y_prob = best_model.predict_proba(X_test)[:, 1]  # Probability estimates for the positive class
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    return accuracy, roc_auc

def logreg_best(X, y):
    # Split the data for time series
    train_size = int(0.6 * len(X))  # 60% for training
    val_size = int(0.2 * len(X))    # 20% for validation
    test_size = len(X) - train_size - val_size  # Remaining 20% for testing

    # Sequential splits
    X_train_val = X[:train_size + val_size]
    y_train_val = y[:train_size + val_size]

    X_test = X[train_size + val_size:]
    y_test = y[train_size + val_size:]

    # Define the Logistic Regression model
    lr_model = LogisticRegression(solver='liblinear')  # 'liblinear' is good for small datasets and binary classification

    # Set up the grid of hyperparameters for tuning
    param_grid = {
        'C': [0.1, 1, 10],  # Regularization strength
        'penalty': ['l2', 'none'],  # Regularization type (no regularization or L2 regularization)
        'max_iter': [100, 200, 300],  # Maximum number of iterations for optimization
        'solver': ['liblinear', 'saga'],  # Optimization algorithms
    }

    # TimeSeriesSplit for time series validation
    tscv = TimeSeriesSplit(n_splits=5)

    # Use GridSearchCV for hyperparameter tuning
    grid_search = GridSearchCV(
        estimator=lr_model,
        param_grid=param_grid,
        scoring='accuracy',  # Use accuracy for evaluation
        cv=tscv,  # Time-series split for validation
        verbose=1,
        n_jobs=-1  # Parallelize for faster computation
    )

    # Fit the model with training and validation data
    grid_search.fit(X_train_val, y_train_val)

    # Get the best model from grid search
    best_model = grid_search.best_estimator_

    # Predict on the test set using the fine-tuned model
    y_pred = best_model.predict(X_test)

    # Evaluate the model's accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Calculate ROC AUC score
    y_prob = best_model.predict_proba(X_test)[:, 1]  # Probability estimates for the positive class
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    return accuracy, roc_auc



## Importing data and creating sentiment index

In [ ]:
cc = pd.read_csv('/Users/gabo/Documents/Thesis/OpenAI/Comments/BERT/CryptoCurrency_merged.csv')
bb = pd.read_csv('/Users/gabo/Documents/Thesis/OpenAI/Comments/BERT/Bitcoin_merged.csv')
merged_sentiment = pd.concat([cc, bb], ignore_index=True)
merged_sentiment['date'] = pd.to_datetime(merged_sentiment['date'], errors='coerce')
merged_sentiment = merged_sentiment.sort_values(by='date').reset_index(drop=True)
valid_labels = ['bullish', 'neutral', 'bearish']
df = merged_sentiment[merged_sentiment['prediction_bert'].isin(valid_labels) & merged_sentiment['prediction_gpt'].isin(valid_labels)]
# Load and clean data
df['date'] = pd.to_datetime(df['date'], errors='coerce')
start_date = min(df['date']) - dt.timedelta(days=2)
end_date = max(df['date']) + dt.timedelta(days=1)
ticker = "BTC-USD"
alpha = 0.1
asset = 'Bitcoin'
financial_data = download_and_prepare_data(ticker, start_date, end_date)

In [ ]:
#Import data from yfinance
daily_sentiment_df = daily_sentiment(df, model='gpt',index='no_neutral')
merged_df = calculate_sentiment_metrics(financial_data, daily_sentiment_df)
merged_df['Date'] = pd.to_datetime(merged_df['Date'], errors='coerce')
visualize_correlation(asset, ticker, merged_df,sentiment_column='sentiment_index')


## Evaluating Trading Strategies

In [ ]:
# Clear metrics_dict for this run
metrics_dict = {}

# Calculate Sharpe ratios for different strategies
run_and_store_metrics('Asset_Sentiment_MA', merged_df, 'sentiment_index', 50, 200, plot=False)
run_and_store_metrics('Price_Driven_MA', merged_df, 'Adj Close', 50, 200, plot=False)
run_and_store_metrics('Buy_and_Hold', merged_df, 'Adj Close', 50, 200, plot=False)

# Convert metrics_dict to DataFrame
strategies_metrics_df = pd.DataFrame.from_dict(metrics_dict, orient='index')
strategies_metrics_df

## Cointegration testing

In [ ]:
cointegration_testing(merged_df, 'Adj Close', 'sentiment_index')

## Classification models

In [ ]:
columns = ['logreturn','sentiment_index','positive_logreturn','positive_sentiment']

filtered_df = merged_df[columns]
filtered_df = create_diff_lags(filtered_df,length=15,diff=False)
filtered_df.dropna()

possible_columns = filtered_df.columns[4:]
sentiment_features = [col for col in filtered_df.columns[4:] if 'sentiment_index' in col]
other_features = [col for col in filtered_df.columns[4:] if col not in sentiment_features]

y = filtered_df['positive_logreturn']


In [ ]:

# Define the looping parameters
models = [ 'gpt,''bert']
indices = [ 'simple','no_neutral','ln']

# Initialize lists to store results
# results_accuracy = []
# results_roc = []

# Loop through models and indices
for model in models:
    for index in indices:
        # Generate daily sentiment dataframe
        daily_sentiment_df = daily_sentiment(
            df,
            model=model,
            index=index,
            correction='none',
            alpha=alpha,
            volume_adjustment=False
        )

        # Merge with financial data and calculate sentiment metrics
        merged_df = calculate_sentiment_metrics(financial_data, daily_sentiment_df)
        merged_df['Date'] = pd.to_datetime(merged_df['Date'], errors='coerce')

        # Select relevant columns and create lagged features
        columns = ['logreturn', 'sentiment_index', 'positive_logreturn', 'positive_sentiment']
        filtered_df = merged_df[columns]
        filtered_df = create_diff_lags(filtered_df, length=10, diff=False)
        filtered_df.dropna(inplace=True)

        # Define the target and feature set
        y = filtered_df['positive_sentiment']
        possible_columns = filtered_df.columns[4:]
        X = filtered_df[possible_columns]
        sentiment_features = [col for col in filtered_df.columns[4:] if 'sentiment_index' in col]
        other_features = [col for col in filtered_df.columns[4:] if col not in sentiment_features]

        # Evaluate models and get best feature combinations
        model_results = evaluate_models_lags()
        x_list_xgb = model_results['XGBoost-RF']['best_combination']
        x_list_svm = model_results['SVM']['best_combination']
        x_list_logreg = model_results['Logistic Regression']['best_combination']

        # Model evaluations
        xgb_accuracy, xgb_roc = xgbrf_best(X[np.array(x_list_xgb)], y)
        svm_accuracy, svm_roc = svm_best(X[np.array(x_list_svm)], y)
        logreg_accuracy, logreg_roc = logreg_best(X[np.array(x_list_logreg)], y)

        # Record results
        results_accuracy.append({
            'Model': model,
            'Index': index,
            'XGBoost_Accuracy': xgb_accuracy,
            'SVM_Accuracy': svm_accuracy,
            'LogReg_Accuracy': logreg_accuracy
        })

        results_roc.append({
            'Model': model,
            'Index': index,
            'XGBoost_ROC': xgb_roc,
            'SVM_ROC': svm_roc,
            'LogReg_ROC': logreg_roc
        })

# Create dataframes from results
accuracy_df = pd.DataFrame(results_accuracy)
roc_df = pd.DataFrame(results_roc)

# Display or save the dataframes
print("Accuracy Results:")
print(accuracy_df)

print("\nROC Results:")
print(roc_df)
